# gkmsvm-lite Application Note Figures

Generates publication-quality figures for the Bioinformatics application note.

**Inputs** (from `benchmarks/results/`):
- `timing_vs_lsgkm.tsv` — wall-clock comparison vs LS-GKM C
- `bench_dsqtl.tsv` — dsQTL VEP (deltaSVM, kernel, gkmexplain)
- `task4_*.tsv` — DART-Eval Task 4 chromatin activity
- `task5_*.tsv` — DART-Eval Task 5 variant effect prediction

**Outputs** (to `benchmarks/figures/`):
- `fig1_performance.pdf` — training + inference speed vs LS-GKM C
- `fig2_dart_eval.pdf` — DART-Eval accuracy benchmarks
- `fig3_gkmexplain.pdf` — GkmExplain validation + VEP method comparison
- `table1_summary.tsv` — summary statistics table

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Bioinformatics journal style
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

RESULTS = Path("results")
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

# Colors
C_LSGKM = "#888888"     # gray — LS-GKM C baseline
C_CPU   = "#4682B4"     # steelblue — gkmsvm-lite CPU
C_GPU   = "#E05555"     # coral-red — gkmsvm-lite CUDA
C_DSVM  = "#66A61E"     # green — deltaSVM
C_KERN  = "#E6AB02"     # amber — kernel VEP
C_EXPL  = "#7570B3"     # purple — gkmexplain VEP

# Double-column width for Bioinformatics: 178 mm = 7.0 in
FIG_WIDE = (7.0, 2.8)
FIG_TALL = (7.0, 5.0)

## Figure 1 — Performance comparison: gkmsvm-lite vs LS-GKM C

In [ ]:
def load_timing_tsv(path):
    """Parse the two-section timing TSV from timing_vs_lsgkm.py."""
    train_rows, infer_rows = [], []
    section = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("## Training"):
                section = "train"
                continue
            elif line.startswith("## Inference"):
                section = "infer"
                continue
            elif line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "train" and parts[0] != "N":
                train_rows.append(parts)
            elif section == "infer" and parts[0] != "N_train":
                infer_rows.append(parts)

    def to_float(v):
        return np.nan if v == "NA" else float(v)

    train = pd.DataFrame(train_rows, columns=[
        "N", "seq_len", "gram_gb", "t_lsgkm", "t_cpu", "t_gpu",
        "n_sv_lsgkm", "n_sv_lite",
    ]).apply(lambda c: c.map(to_float))

    infer = pd.DataFrame(infer_rows, columns=[
        "N_train", "n_sv", "n_test", "t_lsgkm", "t_cpu", "t_gpu",
    ]).apply(lambda c: c.map(to_float))

    return train, infer


# Load data
timing_files = sorted(RESULTS.glob("timing_vs_lsgkm*.tsv"))
if timing_files:
    timing_path = timing_files[-1]  # most recent
    print(f"Loading: {timing_path}")
    df_train, df_infer = load_timing_tsv(timing_path)
    display(df_train)
    display(df_infer)
else:
    print("No timing results found. Run timing_vs_lsgkm.py first.")
    df_train, df_infer = None, None

In [ ]:
if df_train is not None:
    fig, axes = plt.subplots(1, 2, figsize=FIG_WIDE)

    # ── Panel A: Training time ──
    ax = axes[0]
    ax.set_title("A", loc="left", fontweight="bold", fontsize=10)

    mask = df_train["t_lsgkm"].notna()
    if mask.any():
        ax.plot(df_train.loc[mask, "N"], df_train.loc[mask, "t_lsgkm"],
                "o-", color=C_LSGKM, label="LS-GKM C", ms=5, lw=1.5)

    ax.plot(df_train["N"], df_train["t_cpu"],
            "s-", color=C_CPU, label="gkmsvm-lite CPU", ms=5, lw=1.5)

    mask = df_train["t_gpu"].notna()
    if mask.any():
        ax.plot(df_train.loc[mask, "N"], df_train.loc[mask, "t_gpu"],
                "^-", color=C_GPU, label="gkmsvm-lite CUDA", ms=5, lw=1.5)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Training set size (N)")
    ax.set_ylabel("Wall-clock time (s)")
    ax.set_title("Training time", fontsize=9, pad=8)
    ax.text(-0.12, 1.05, "A", transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="bottom")
    ax.legend(loc="upper left", frameon=False)
    ax.grid(True, alpha=0.2, which="both")

    # ── Panel B: Inference throughput ──
    ax = axes[1]
    n_test = df_infer["n_test"].iloc[0]

    mask = df_infer["t_lsgkm"].notna()
    if mask.any():
        ax.plot(df_infer.loc[mask, "n_sv"],
                n_test / df_infer.loc[mask, "t_lsgkm"],
                "o-", color=C_LSGKM, label="LS-GKM C", ms=5, lw=1.5)

    ax.plot(df_infer["n_sv"], n_test / df_infer["t_cpu"],
            "s-", color=C_CPU, label="gkmsvm-lite CPU", ms=5, lw=1.5)

    mask = df_infer["t_gpu"].notna()
    if mask.any():
        ax.plot(df_infer.loc[mask, "n_sv"],
                n_test / df_infer.loc[mask, "t_gpu"],
                "^-", color=C_GPU, label="gkmsvm-lite CUDA", ms=5, lw=1.5)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Number of support vectors")
    ax.set_ylabel("Throughput (sequences/s)")
    ax.set_title("Inference throughput", fontsize=9, pad=8)
    ax.text(-0.12, 1.05, "B", transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="bottom")
    ax.legend(loc="upper right", frameon=False)
    ax.grid(True, alpha=0.2, which="both")

    plt.tight_layout()
    fig.savefig(FIGURES / "fig1_performance.pdf")
    fig.savefig(FIGURES / "fig1_performance.png")
    plt.show()

    # Print speedup summary
    for _, r in df_train.iterrows():
        if pd.notna(r["t_lsgkm"]) and pd.notna(r["t_gpu"]):
            print(f"N={r['N']:.0f}: training speedup "
                  f"{r['t_lsgkm']/r['t_gpu']:.1f}x (GPU vs C)")
    for _, r in df_infer.iterrows():
        if pd.notna(r["t_lsgkm"]) and pd.notna(r["t_gpu"]):
            print(f"N_SV={r['n_sv']:.0f}: inference speedup "
                  f"{r['t_lsgkm']/r['t_gpu']:.1f}x (GPU vs C)")

## Figure 2 — DART-Eval benchmarks

In [ ]:
# Reference values from Pampari et al. 2024 (NeurIPS, arXiv:2412.05430)
# Table 2 (Task 4) and Table 3/Fig 6 (Task 5)
DART_REF = {
    "task4": {
        # Method: (mean AUROC across cell lines, source)
        "Probed DNABERT-2":  0.784,
        "Probed NT-v2":     0.791,
        "Probed HyenaDNA":  0.800,
        "Fine-tuned NT-v2": 0.935,
        "ChromBPNet":       0.946,
    },
    "task5_dsqtl": {
        # Method: (AUPRC, source)
        "gkm-SVM (pub)": 0.19,    # Fig 6, deltaSVM DNase 68M
        "Enformer":      0.33,
        "ChromBPNet":    0.43,
    },
}

# Load our DART-Eval Task 4 results
task4_files = sorted(RESULTS.glob("task4_*.tsv"))
task4_results = []
for f in task4_files:
    try:
        df = pd.read_csv(f, sep="\t")
        task4_results.append(df)
        print(f"Loaded: {f.name}  ({len(df)} rows)")
    except Exception as e:
        print(f"Skip {f.name}: {e}")

if task4_results:
    df_task4 = pd.concat(task4_results, ignore_index=True)
    display(df_task4)
else:
    df_task4 = None
    print("No Task 4 results found. Run dart_eval.py --task activity first.")

# Load our dsQTL VEP results
dsqtl_files = sorted(RESULTS.glob("bench_dsqtl*.tsv"))
if dsqtl_files:
    df_dsqtl = pd.read_csv(dsqtl_files[-1], sep="\t")
    print(f"\nLoaded: {dsqtl_files[-1].name}")
    display(df_dsqtl)
else:
    df_dsqtl = None
    print("No dsQTL VEP results found. Run bench_dsqtl.py first.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIG_WIDE)

# ── Panel A: Task 4 — Chromatin activity AUROC ──
ax = axes[0]

# Collect methods + AUROCs for bar chart
methods_a, aurocs_a, colors_a = [], [], []

# Our results
if df_task4 is not None and "auroc" in df_task4.columns:
    our_auroc = df_task4["auroc"].mean()
    methods_a.append("gkmsvm-lite")
    aurocs_a.append(our_auroc)
    colors_a.append(C_CPU)
else:
    # Placeholder from our earlier GM12878 run
    methods_a.append("gkmsvm-lite")
    aurocs_a.append(0.855)
    colors_a.append(C_CPU)

# Reference methods
ref_colors = ["#AAAAAA"] * len(DART_REF["task4"])
for method, auroc in DART_REF["task4"].items():
    methods_a.append(method)
    aurocs_a.append(auroc)
    colors_a.append("#BBBBBB")

y_pos = np.arange(len(methods_a))
bars = ax.barh(y_pos, aurocs_a, color=colors_a, height=0.6, edgecolor="white")
bars[0].set_color(C_CPU)
bars[0].set_edgecolor(C_CPU)

ax.set_yticks(y_pos)
ax.set_yticklabels(methods_a)
ax.set_xlabel("AUROC")
ax.set_xlim(0.7, 1.0)
ax.set_title("Task 4: Chromatin activity", fontsize=9, pad=8)
ax.text(-0.35, 1.05, "A", transform=ax.transAxes,
        fontsize=11, fontweight="bold", va="bottom")
ax.invert_yaxis()

for i, v in enumerate(aurocs_a):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=6.5)

# ── Panel B: Task 5 — dsQTL VEP AUPRC ──
ax = axes[1]

methods_b, auprcs_b, colors_b = [], [], []

# Our results — three VEP methods
if df_dsqtl is not None and "auprc" in df_dsqtl.columns:
    for _, row in df_dsqtl.iterrows():
        label = row.get("method", row.get("name", str(row.name)))
        methods_b.append(f"gkmsvm-lite\n{label}")
        auprcs_b.append(row["auprc"])
        if "delta" in str(label).lower():
            colors_b.append(C_DSVM)
        elif "kernel" in str(label).lower():
            colors_b.append(C_KERN)
        elif "explain" in str(label).lower():
            colors_b.append(C_EXPL)
        else:
            colors_b.append(C_CPU)
else:
    # Placeholders — update after running bench_dsqtl.py
    for method, color in [("deltaSVM", C_DSVM), ("kernel", C_KERN),
                          ("gkmexplain", C_EXPL)]:
        methods_b.append(f"gkmsvm-lite\n{method}")
        auprcs_b.append(np.nan)
        colors_b.append(color)

# Reference methods
for method, auprc in DART_REF["task5_dsqtl"].items():
    methods_b.append(method)
    auprcs_b.append(auprc)
    colors_b.append("#BBBBBB")

y_pos = np.arange(len(methods_b))
valid = [not np.isnan(v) for v in auprcs_b]
bars = ax.barh(y_pos, [v if ok else 0 for v, ok in zip(auprcs_b, valid)],
               color=colors_b, height=0.6, edgecolor="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(methods_b)
ax.set_xlabel("AUPRC")
ax.set_xlim(0, 0.6)
ax.set_title("Task 5: dsQTL variant effects", fontsize=9, pad=8)
ax.text(-0.35, 1.05, "B", transform=ax.transAxes,
        fontsize=11, fontweight="bold", va="bottom")
ax.invert_yaxis()

for i, (v, ok) in enumerate(zip(auprcs_b, valid)):
    if ok:
        ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=6.5)
    else:
        ax.text(0.01, i, "(pending)", va="center", fontsize=6, color="gray")

plt.tight_layout()
fig.savefig(FIGURES / "fig2_dart_eval.pdf")
fig.savefig(FIGURES / "fig2_dart_eval.png")
plt.show()

## Figure 3 — GkmExplain validation and VEP method comparison

In [ ]:
# Panel A: GkmExplain completeness axiom (run inline with a small model)
# Panel B: VEP method comparison scatter (deltaSVM vs kernel vs gkmexplain)
#
# Panel A demonstrates a theoretical property; Panel B shows practical
# agreement between the three scoring methods.

from gkmsvm import one_hot_encode, train_gkmsvm
from gkmsvm.explain import gkmexplain

# Build a small model for the completeness demo
import random
rng = random.Random(42)
seqs_pos = ["".join(rng.choice("ACGT") for _ in range(50)) for _ in range(200)]
seqs_neg = ["".join(rng.choice("ACGT") for _ in range(50)) for _ in range(200)]

demo_model = train_gkmsvm(
    seqs_pos, seqs_neg,
    kernel_type="estimated", l=11, k=7, d=3, C=1.0,
    verbose=False,
)
print(f"Demo model: {demo_model.num_support_vectors} SVs")

# Score + attribute a batch
test_seqs = ["".join(rng.choice("ACGT") for _ in range(50)) for _ in range(100)]
X = np.stack([one_hot_encode(s) for s in test_seqs])

scores = demo_model(X).squeeze(-1)
attr = gkmexplain(demo_model, X, mode=0)
attr_sum = attr.sum(axis=(1, 2))
expected = scores - demo_model.bias

residuals = attr_sum - expected
print(f"Completeness: max|residual| = {np.abs(residuals).max():.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIG_WIDE)

# ── Panel A: Completeness axiom ──
ax = axes[0]

ax.scatter(expected, attr_sum, s=12, alpha=0.6, color=C_CPU,
           edgecolors="none", rasterized=True)

lo = min(expected.min(), attr_sum.min())
hi = max(expected.max(), attr_sum.max())
margin = (hi - lo) * 0.05
ax.plot([lo - margin, hi + margin], [lo - margin, hi + margin],
        "k--", lw=0.8, alpha=0.4)

r = np.corrcoef(expected, attr_sum)[0, 1]
max_err = np.abs(residuals).max()
ax.text(0.05, 0.92, f"r = {r:.6f}\nmax |err| = {max_err:.1e}",
        transform=ax.transAxes, fontsize=7, va="top",
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

ax.set_xlabel(r"$f(x) - b$  (SVM decision value)")
ax.set_ylabel(r"$\sum$ GkmExplain mode-0 attributions")
ax.set_title("Completeness axiom", fontsize=9, pad=8)
ax.text(-0.15, 1.05, "A", transform=ax.transAxes,
        fontsize=11, fontweight="bold", va="bottom")

# ── Panel B: VEP method comparison ──
ax = axes[1]

if df_dsqtl is not None and "auprc" in df_dsqtl.columns:
    # Plot from actual results
    for _, row in df_dsqtl.iterrows():
        label = row.get("method", row.get("name", ""))
        ax.bar(label, row["auprc"])
    ax.set_ylabel("AUPRC")
    ax.set_title("dsQTL VEP: method comparison", fontsize=9, pad=8)
else:
    # Schematic placeholder
    methods = ["deltaSVM", "kernel\nVEP", "gkmexplain\nVEP"]
    colors = [C_DSVM, C_KERN, C_EXPL]
    descriptions = [
        r"$\sum \Delta w_{kmer}$" + "\n(linear approx.)",
        r"$f(alt) - f(ref)$" + "\n(exact, 2 evals)",
        r"$h_{alt} - h_{ref}$" + "\n(1 explain call)",
    ]
    x = np.arange(len(methods))
    bars = ax.bar(x, [0.3, 0.3, 0.3], color=colors, width=0.5,
                  edgecolor="white", alpha=0.3)

    for i, desc in enumerate(descriptions):
        ax.text(i, 0.15, desc, ha="center", va="center", fontsize=6.5)
    ax.text(0.5, 0.85, "Run bench_dsqtl.py\nto populate",
            transform=ax.transAxes, ha="center", fontsize=8,
            color="gray", style="italic")

    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.set_ylabel("AUPRC")
    ax.set_ylim(0, 0.5)
    ax.set_title("dsQTL VEP: method comparison", fontsize=9, pad=8)

ax.text(-0.15, 1.05, "B", transform=ax.transAxes,
        fontsize=11, fontweight="bold", va="bottom")

plt.tight_layout()
fig.savefig(FIGURES / "fig3_gkmexplain.pdf")
fig.savefig(FIGURES / "fig3_gkmexplain.png")
plt.show()

## Summary table

In [ ]:
# Collate key numbers into a summary table for the paper text
summary_rows = []

# Timing
if df_train is not None:
    for _, r in df_train.iterrows():
        row = {"benchmark": "training", "N": int(r["N"])}
        if pd.notna(r["t_lsgkm"]):
            row["lsgkm_c_s"] = f"{r['t_lsgkm']:.1f}"
        row["lite_cpu_s"] = f"{r['t_cpu']:.1f}"
        if pd.notna(r["t_gpu"]):
            row["lite_gpu_s"] = f"{r['t_gpu']:.1f}"
            if pd.notna(r["t_lsgkm"]):
                row["gpu_speedup"] = f"{r['t_lsgkm']/r['t_gpu']:.1f}x"
        summary_rows.append(row)

if df_infer is not None:
    for _, r in df_infer.iterrows():
        row = {"benchmark": "inference", "N": int(r["n_sv"])}
        if pd.notna(r["t_lsgkm"]):
            row["lsgkm_c_s"] = f"{r['t_lsgkm']:.2f}"
        row["lite_cpu_s"] = f"{r['t_cpu']:.2f}"
        if pd.notna(r["t_gpu"]):
            row["lite_gpu_s"] = f"{r['t_gpu']:.2f}"
            if pd.notna(r["t_lsgkm"]):
                row["gpu_speedup"] = f"{r['t_lsgkm']/r['t_gpu']:.1f}x"
        summary_rows.append(row)

# DART-Eval
if df_task4 is not None and "auroc" in df_task4.columns:
    for _, r in df_task4.iterrows():
        cell = r.get("cell_line", "unknown")
        summary_rows.append({
            "benchmark": f"task4_{cell}",
            "auroc": f"{r['auroc']:.4f}",
            "auprc": f"{r.get('auprc', np.nan):.4f}" if pd.notna(r.get('auprc')) else "",
        })

if summary_rows:
    df_summary = pd.DataFrame(summary_rows)
    display(df_summary)
    df_summary.to_csv(FIGURES / "table1_summary.tsv", sep="\t", index=False)
    print(f"Saved: {FIGURES / 'table1_summary.tsv'}")
else:
    print("No results to summarize yet.")

## Supplementary: Throughput sweep heatmap

In [ ]:
# Run throughput_sweep.py with --device both, then load results here.
# For now, this generates the sweep inline if the model fixture exists.

from pathlib import Path
fixture = Path("../tests/fixtures/encode_ENCFF579AOX.model.txt.gz")

if fixture.exists():
    from gkmsvm.importers.lsgkm import load_lsgkm_model
    from gkmsvm.codec import one_hot_encode as ohe
    import time

    model = load_lsgkm_model(str(fixture))
    print(f"Model: {model.num_support_vectors} SVs")

    seq_lens = [50, 100, 200, 300]
    batch_sizes = [1, 4, 16, 64, 256]

    results = np.zeros((len(seq_lens), len(batch_sizes)))

    for i, sl in enumerate(seq_lens):
        for j, bs in enumerate(batch_sizes):
            rng_np = np.random.default_rng(42)
            x = np.zeros((bs, 4, sl), dtype=np.float64)
            idx = rng_np.integers(0, 4, size=(bs, sl))
            for b in range(bs):
                x[b, idx[b], np.arange(sl)] = 1.0

            _ = model(x)  # warmup
            n_iters = max(1, 64 // bs)
            t0 = time.perf_counter()
            for _ in range(n_iters):
                _ = model(x)
            elapsed = time.perf_counter() - t0
            results[i, j] = (n_iters * bs) / elapsed
            print(f"  L={sl:3d} B={bs:3d}: {results[i,j]:.0f} seq/s")

    fig, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(results, aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(batch_sizes)))
    ax.set_xticklabels(batch_sizes)
    ax.set_yticks(range(len(seq_lens)))
    ax.set_yticklabels(seq_lens)
    ax.set_xlabel("Batch size")
    ax.set_ylabel("Sequence length (bp)")
    ax.set_title(f"CPU throughput (seq/s), {model.num_support_vectors} SVs")

    for i in range(len(seq_lens)):
        for j in range(len(batch_sizes)):
            ax.text(j, i, f"{results[i,j]:.0f}", ha="center", va="center",
                    fontsize=6.5, color="white" if results[i,j] > results.max()*0.6 else "black")

    plt.colorbar(im, ax=ax, label="seq/s", shrink=0.8)
    plt.tight_layout()
    fig.savefig(FIGURES / "supp_throughput_heatmap.pdf")
    fig.savefig(FIGURES / "supp_throughput_heatmap.png")
    plt.show()
else:
    print(f"Model fixture not found at {fixture}")
    print("Run throughput_sweep.py separately and load results.")